# 8. Backtest — QA

**Purpose** — Validate backtest internal consistency and make assumptions explicit.

**Source tables**
- `py_backtest_summary`
- `py_backtest_results`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## Assumptions (declared per scenario)

For every scenario in `py_backtest_summary`, the following must be documented (or audited as defaults):

| Assumption | Default if absent |
|---|---|
| Fee model      | Pool fee_tier × volume × share_of_pool |
| Price source   | `mart_price_analytics_daily` close |
| Liquidity      | Constant initial_liquidity over window (no rebalance) |
| Rebalance rule | None (static) |
| Gas model      | Not included (gas=0) |
| Slippage       | Not modelled |
| Capital alloc. | `capital_committed_token0` per scenario |


## §1 — Data fetch & coverage

In [3]:
summary = db.read_query("SELECT * FROM py_backtest_summary")
results = db.read_query(f"SELECT * FROM py_backtest_results WHERE day <= toDate('{END_DATE}') ORDER BY scenario_id, day")
print(f"scenarios: {summary.height}")
print(f"daily results: {results.height}")
summary.select("scenario_id", "pool", "start_block", "end_block", "tick_lower", "tick_upper", "capital_committed_token0", "rebalance_strategy")

scenarios: 1
daily results: 60


scenario_id,pool,start_block,end_block,tick_lower,tick_upper,capital_committed_token0,rebalance_strategy
str,str,i64,i64,i64,i64,f64,str
"""0x88e6a0c2ddd26feeb64f039a2c41…","""0x88e6a0c2ddd26feeb64f039a2c41…",24400000,24820000,197000,203000,100.0,"""none"""


## §2 — Sanity checks

In [4]:
# Per-scenario assumption logging completeness
n_missing = summary.filter(
    pl.col("scenario_id").is_null() | pl.col("rebalance_strategy").is_null()
).height
qa_check("scenario_id and rebalance_strategy non-null", n_missing == 0,
         detail=f"missing={n_missing}")

# Range validity
n_bad = summary.filter(pl.col("tick_lower") >= pl.col("tick_upper")).height
qa_check("backtest summary: tick_lower < tick_upper", n_bad == 0,
         detail=f"violations={n_bad}")

# Capital conservation: at t=0, principal_value_token0 should match capital_committed_token0
recon = []
for sid in summary["scenario_id"].to_list():
    s_capital = float(summary.filter(pl.col("scenario_id") == sid)["capital_committed_token0"][0])
    init_row = results.filter(pl.col("scenario_id") == sid).sort("day").head(1)
    if init_row.height:
        init_principal = float(init_row["principal_value_token0"][0])
        recon.append({"scenario_id": sid, "committed": s_capital, "init_principal": init_principal,
                      "rel_resid": abs(init_principal - s_capital) / max(s_capital, 1e-6)})
recon_df = pl.DataFrame(recon)
n_off = recon_df.filter(pl.col("rel_resid") > 0.05).height
qa_check("initial principal_value_token0 ≈ capital_committed_token0 (±5%)",
         n_off == 0, severity="warn",
         detail=f"scenarios_off={n_off}/{recon_df.height}")
recon_df

🟢 PASS | scenario_id and rebalance_strategy non-null  —  missing=0
🟢 PASS | backtest summary: tick_lower < tick_upper  —  violations=0
🟢 PASS | initial principal_value_token0 ≈ capital_committed_token0 (±5%)  —  scenarios_off=0/1


scenario_id,committed,init_principal,rel_resid
str,f64,f64,f64
"""0x88e6a0c2ddd26feeb64f039a2c41…",100.0,99.906382,0.000936


In [5]:
# Non-negativity
n_neg_lp  = results.filter(pl.col("lp_value_including_fees_token0") < 0).height
n_neg_fee = results.filter(pl.col("fees_accrued_token0_equiv_cumulative") < 0).height
qa_check("LP value never negative", n_neg_lp == 0, detail=f"violations={n_neg_lp}")
qa_check("cumulative fees never negative", n_neg_fee == 0, detail=f"violations={n_neg_fee}")

# Cumulative fees monotonic
mono_q = f"""
SELECT count() AS n FROM (
  SELECT scenario_id, day, fees_accrued_token0_equiv_cumulative,
         lagInFrame(fees_accrued_token0_equiv_cumulative, 1, toFloat64(0)) OVER (PARTITION BY scenario_id ORDER BY day) AS prev_f
  FROM py_backtest_results
)
WHERE prev_f > fees_accrued_token0_equiv_cumulative
"""
n_mono = int(db.read_query(mono_q)["n"][0])
qa_check("cumulative fees monotonic per scenario", n_mono == 0, detail=f"violations={n_mono}")


🟢 PASS | LP value never negative  —  violations=0
🟢 PASS | cumulative fees never negative  —  violations=0
🟢 PASS | cumulative fees monotonic per scenario  —  violations=0


In [6]:
# HODL benchmark consistency: per scenario, hodl_value_token0 at t=end ≈ summary.hodl_end_value_token0.
# NOTE: summary HODL is anchored on the last swap at/before end_block; results last row is the
# last calendar day's close. Tolerance widened to 5% to absorb that anchor difference.
hodl_rows = []
for sid in summary["scenario_id"].to_list():
    s_hodl_end = float(summary.filter(pl.col("scenario_id") == sid)["hodl_end_value_token0"][0])
    last = results.filter(pl.col("scenario_id") == sid).sort("day").tail(1)
    if last.height:
        last_hodl = float(last["hodl_value_token0"][0])
        hodl_rows.append({"scenario_id": sid, "summary_hodl_end": s_hodl_end, "results_last_hodl": last_hodl,
                          "rel_resid": abs(s_hodl_end - last_hodl) / max(s_hodl_end, 1e-6)})
hodl_df = pl.DataFrame(hodl_rows)
n_off = hodl_df.filter(pl.col("rel_resid") > 0.05).height
qa_check("HODL final values reconcile (block-anchor vs day-anchor, ±5%)",
         n_off == 0, severity="warn",
         detail=f"scenarios_off={n_off}/{hodl_df.height}")
hodl_df


🟢 PASS | HODL final values reconcile (block-anchor vs day-anchor, ±5%)  —  scenarios_off=0/1


scenario_id,summary_hodl_end,results_last_hodl,rel_resid
str,f64,f64,f64
"""0x88e6a0c2ddd26feeb64f039a2c41…",102.142969,101.007348,0.011118


## §3 — Diagnostic visuals

**QA purpose**: assumption box + scenario summary table — make the input space explicit.

In [7]:
display_cols = ["scenario_id", "tick_lower", "tick_upper", "capital_committed_token0",
                "rebalance_strategy", "swap_count_in_window", "fees_accrued_token0_equiv_generated",
                "impermanent_loss_token0", "pnl_vs_initial_token0", "pnl_vs_hodl_token0", "observed_time_in_range_pct"]
summary.select(display_cols)

scenario_id,tick_lower,tick_upper,capital_committed_token0,rebalance_strategy,swap_count_in_window,fees_accrued_token0_equiv_generated,impermanent_loss_token0,pnl_vs_initial_token0,pnl_vs_hodl_token0,observed_time_in_range_pct
str,i64,i64,f64,str,i64,f64,f64,f64,f64,f64
"""0x88e6a0c2ddd26feeb64f039a2c41…",197000,203000,100.0,"""none""",394593,4.542394,-0.162935,6.522427,4.379459,1.0


**QA purpose**: LP value vs HODL evolution per scenario. Divergence visible immediately.

In [8]:
for sid in summary["scenario_id"].to_list():
    sub = results.filter(pl.col("scenario_id") == sid).sort("day")
    if sub.height < 2:
        continue
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=(f"Scenario {sid}: LP vs HODL", "PnL decomposition"))
    fig.add_trace(go.Scatter(x=sub["day"].to_list(), y=sub["lp_value_including_fees_token0"].to_list(), name="LP value", line=dict(color="#3b82f6")), row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["day"].to_list(), y=sub["hodl_value_token0"].to_list(), name="HODL", line=dict(color="#9ca3af", dash="dot")), row=1, col=1)
    fig.add_trace(go.Bar(x=sub["day"].to_list(), y=sub["fees_accrued_token0_equiv_cumulative"].to_list(), name="cum fees", marker_color="#22c55e"), row=2, col=1)
    fig.add_trace(go.Bar(x=sub["day"].to_list(), y=sub["impermanent_loss_token0"].to_list(), name="IL", marker_color="#ef4444"), row=2, col=1)
    fig.update_layout(height=480, title_text=f"Scenario {sid}", barmode="overlay")
    fig.show()

**QA purpose**: capital conservation chart — LP value annotated with initial capital; never below zero.

In [9]:
fig = go.Figure()
for sid in summary["scenario_id"].to_list():
    sub = results.filter(pl.col("scenario_id") == sid).sort("day")
    if sub.height < 2:
        continue
    fig.add_trace(go.Scatter(x=sub["day"].to_list(), y=sub["lp_value_including_fees_token0"].to_list(), name=f"{sid}", mode="lines"))
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="zero floor")
fig.update_layout(height=360, title_text="LP value over time — never below zero")
fig.show()

## §4 — QA verdict

In [10]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: PASS
  PASS: 7   WARN: 0   FAIL: 0


label,status,severity,detail
str,str,str,str
"""scenario_id and rebalance_stra…","""PASS""","""fail""","""missing=0"""
"""backtest summary: tick_lower <…","""PASS""","""fail""","""violations=0"""
"""initial principal_value_token0…","""PASS""","""warn""","""scenarios_off=0/1"""
"""LP value never negative""","""PASS""","""fail""","""violations=0"""
"""cumulative fees never negative""","""PASS""","""fail""","""violations=0"""
"""cumulative fees monotonic per …","""PASS""","""fail""","""violations=0"""
"""HODL final values reconcile (b…","""PASS""","""warn""","""scenarios_off=0/1"""
